<a href="https://colab.research.google.com/github/satyamroy3913-maker/GenAi_Task/blob/main/Gemini_structured_JSON_extraction_project0301.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cell 1

In [ ]:
!pip install -q reportlab google-genai pydantic

import os, sys, json, time
try:
    from google.colab import userdata
    os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")
except Exception:
    from getpass import getpass
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

os.makedirs("/content/extractor/samples", exist_ok=True)
%cd /content/extractor
sys.path.insert(0, "/content/extractor")

Cell 2: schema.py

In [ ]:
%%writefile schema.py
from enum import Enum
from typing import List, Optional
from pydantic import BaseModel, Field

class InvoiceStatus(str, Enum):          # enum
    PAID = "PAID"
    UNPAID = "UNPAID"
    OVERDUE = "OVERDUE"

class Address(BaseModel):                # nested object
    street: str
    city: str
    pin: str = Field(description="6-digit PIN code as a string")

class LineItem(BaseModel):               # list item
    description: str
    quantity: float
    unit_price: float
    amount: float

class Invoice(BaseModel):
    invoice_number: str
    invoice_date: str = Field(description="Date in ISO format YYYY-MM-DD")
    vendor_name: str
    vendor_address: Address
    bill_to_name: str
    currency: str = Field(description="3-letter code like INR or USD")
    line_items: List[LineItem]
    subtotal: float
    tax_amount: float
    total_amount: float
    status: InvoiceStatus
    po_number: Optional[str] = Field(
        default=None, description="Purchase order number. null if not present on the document")

Cell 3: llm.py (timeout + retry + model fallback)

In [ ]:
%%writefile llm.py
import os, time
from google import genai
from google.genai import types
from google.genai.errors import ServerError, ClientError

PREFERRED = ["gemini-3.8-flash"]
_client, _models = None, None

def client():
    global _client
    if _client is None:
        _client = genai.Client(
            api_key=os.environ["GEMINI_API_KEY"],
            http_options=types.HttpOptions(timeout=60_000))   # 60s timeout
    return _client

def models():
    """Preferred model first, then other flash models available on this key."""
    global _models
    if _models is None:
        found = []
        try:
            for m in client().models.list():
                n = m.name.replace("models/", "")
                ok = "generateContent" in (m.supported_actions or [])
                bad = any(x in n for x in ["image", "tts", "live", "audio", "embedding"])
                if "flash" in n and ok and not bad and n not in PREFERRED:
                    found.append(n)
        except Exception as e:
            print("model list failed:", e)
        _models = (PREFERRED + sorted(found, reverse=True))[:4]
        print("Models to try:", _models)
    return _models

def call(contents, config, retries=3):
    last = None
    for model in models():
        for attempt in range(retries):
            try:
                return client().models.generate_content(
                    model=model, contents=contents, config=config), model
            except ServerError as e:                       # 500/503
                last = e; wait = 5 * 2 ** attempt
                print(f"  [{model}] server {e.code}, retry in {wait}s")
                time.sleep(wait)
            except ClientError as e:
                last = e
                if e.code == 429:                          # rate limit
                    wait = 20 * (attempt + 1)
                    print(f"  [{model}] rate limit, retry in {wait}s")
                    time.sleep(wait)
                else:                                      # 404/400: try next model
                    print(f"  [{model}] {e.code}, trying next model")
                    break
            except Exception as e:                         # timeout / network
                last = e
                print(f"  [{model}] {type(e).__name__}, retry")
                time.sleep(5)
    raise RuntimeError(f"All models failed. Last error: {last}")

Cell 4: main.py (CLI)

In [ ]:
%%writefile main.py
import sys, mimetypes
import json
from google.genai import types
from schema import Invoice
from llm import call

PROMPT = ("Extract the invoice data from this document. "
          "Dates must be YYYY-MM-DD. Use null for fields that are not present. "
          "Do not guess values that are not in the document.")

def extract(file_path: str) -> dict:
    mime = mimetypes.guess_type(file_path)[0] or "application/pdf"
    with open(file_path, "rb") as f:
        data = f.read()
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=Invoice,
        temperature=0)
    resp, model = call([types.Part.from_bytes(data=data, mime_type=mime), PROMPT], cfg)
    inv = resp.parsed if resp.parsed is not None else Invoice.model_validate_json(resp.text)
    return inv.model_dump(mode="json")

if __name__ == "__main__":
    if len(sys.argv) != 2:
        sys.exit("Usage: python main.py <file_path>")
    print(json.dumps(extract(sys.argv[1]), indent=2))

Cell 5: Synthetic data + ground truth

In [ ]:
from pydantic import TypeAdapter
from google.genai import types
from schema import Invoice
from llm import call

if os.path.exists("predictions.json"):
    os.remove("predictions.json")

def gen_batch(n, cities):
    prompt = f"""Generate {n} diverse fictional B2B invoices.
- Vendors from these Indian cities: {cities}
- 2 to 5 line items each, descriptions under 40 characters
- Include at least one of each status: PAID, UNPAID, OVERDUE
- 1 or 2 invoices must have po_number = null
- 1 invoice in USD, the rest INR
- Dates between 2025-01-01 and 2026-10-31 in YYYY-MM-DD
- Invoice numbers in varied formats (INV-2025-001, 7741/25, AB-00912 ...)"""
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json",
        response_schema=list[Invoice], temperature=1)
    resp, model = call(prompt, cfg)
    items = resp.parsed or TypeAdapter(list[Invoice]).validate_json(resp.text)
    print(f"batch ok ({len(items)} invoices) | model: {model}")
    return items

invoices = []
invoices += gen_batch(5, "Mumbai, Pune, Bengaluru, Hyderabad, Ahmedabad")
invoices += gen_batch(5, "Delhi, Chennai, Kolkata, Jaipur, Surat")
invoices = invoices[:10]

truth = []
for i, inv in enumerate(invoices, 1):
    d = inv.model_dump(mode="json")
    for li in d["line_items"]:
        li["amount"] = round(li["quantity"] * li["unit_price"], 2)
    d["subtotal"] = round(sum(li["amount"] for li in d["line_items"]), 2)
    d["tax_amount"] = round(d["subtotal"] * 0.18, 2)
    d["total_amount"] = round(d["subtotal"] + d["tax_amount"], 2)
    truth.append({"sample_id": f"s{i:02d}", **d})

with open("ground_truth.jsonl", "w") as f:
    for t in truth:
        f.write(json.dumps(t) + "\n")

print(len(truth), "invoices ready")

Cell 6: PDFs render

In [ ]:
from reportlab.lib.pagesizes import A4
from reportlab.pdfgen import canvas
from datetime import datetime

FONTS = ["Helvetica", "Times-Roman", "Courier"]
BOLD  = ["Helvetica-Bold", "Times-Bold", "Courier-Bold"]
FMTS  = ["%Y-%m-%d", "%d %b %Y", "%d/%m/%Y"]     # alag date formats (jaan-boojh kar)

def render(t, path, k):
    f, b, fmt = FONTS[k % 3], BOLD[k % 3], FMTS[k % 3]
    date = datetime.strptime(t["invoice_date"], "%Y-%m-%d").strftime(fmt)
    c = canvas.Canvas(path, pagesize=A4); W, H = A4
    y = H - 60
    c.setFont(b, 20); c.drawString(50, y, "TAX INVOICE" if k % 2 == 0 else "INVOICE")
    c.setFont(f, 10); y -= 30
    a = t["vendor_address"]
    for l in [t["vendor_name"], a["street"], f'{a["city"]} - {a["pin"]}']:
        c.drawString(50, y, l); y -= 14
    y -= 10
    meta = [f'Invoice No: {t["invoice_number"]}', f'Date: {date}',
            f'Bill To: {t["bill_to_name"]}', f'Currency: {t["currency"]}']
    if t["po_number"]:
        meta.insert(3, f'PO Ref: {t["po_number"]}')
    for l in meta:
        c.drawString(50, y, l); y -= 14
    y -= 15
    c.setFont(b, 10)
    for x, h in [(50, "Description"), (300, "Qty"), (350, "Rate"), (450, "Amount")]:
        c.drawString(x, y, h)
    y -= 14; c.setFont(f, 10)
    for li in t["line_items"]:
        c.drawString(50, y, li["description"])
        c.drawString(300, y, f'{li["quantity"]:g}')
        c.drawString(350, y, f'{li["unit_price"]:,.2f}')
        c.drawString(450, y, f'{li["amount"]:,.2f}')
        y -= 14
    y -= 15
    for lab, val in [("Subtotal", t["subtotal"]), ("GST (18%)", t["tax_amount"]), ("Total", t["total_amount"])]:
        c.drawString(350, y, lab); c.drawString(450, y, f"{val:,.2f}"); y -= 14
    y -= 20; c.setFont(b, 14)
    note = {"PAID": "PAID", "OVERDUE": "OVERDUE - past due date",
            "UNPAID": "Payment due within 30 days"}[t["status"]]
    c.drawString(50, y, note)
    c.save()

for k, t in enumerate(truth):
    render(t, f'samples/{t["sample_id"]}.pdf', k)

!ls samples

Cell 7: Ek file par test (CLI)

In [ ]:
!python main.py samples/s01.pdf

Cell 8: Saare 10 samples par run (resume-safe)

In [ ]:
import os, sys, json, time, importlib
from getpass import getpass

os.chdir("/content/extractor"); sys.path.insert(0, "/content/extractor")
if not os.environ.get("GEMINI_API_KEY"):
    os.environ["GEMINI_API_KEY"] = getpass("Paste your Gemini API key: ")

import llm, main
importlib.reload(llm); importlib.reload(main)
from google.genai import types
from google.genai.errors import ServerError, ClientError
from schema import Invoice

# ground truth memory me na ho (restart ke baad) to file se load karo
if "truth" not in globals():
    truth = [json.loads(l) for l in open("ground_truth.jsonl")]

# sirf stable flash models (preview/omni nahi), chalne wala model pehle
BAD = ("preview", "omni", "image", "tts", "live", "audio", "embedding")
cands = ["gemini-3.6-flash"]
try:
    cands += [m for m in llm.models() if m not in cands and not any(b in m for b in BAD)]
except Exception:
    pass
print("Models:", cands)
dead = set()    # jin models ka quota khatam / unavailable hai

def extract_one(path):
    data = open(path, "rb").read()
    cfg = types.GenerateContentConfig(
        response_mime_type="application/json", response_schema=Invoice, temperature=0)
    contents = [types.Part.from_bytes(data=data, mime_type="application/pdf"), main.PROMPT]
    for rnd in range(2):
        limited = False
        for model in cands:
            if model in dead:
                continue
            for attempt in range(3):
                try:
                    r = llm.client().models.generate_content(model=model, contents=contents, config=cfg)
                    inv = r.parsed if r.parsed is not None else Invoice.model_validate_json(r.text)
                    return inv.model_dump(mode="json"), model
                except ServerError:                         # 503/504: thoda ruk ke retry
                    time.sleep(5 * 2 ** attempt)
                except ClientError as e:
                    msg = str(e)
                    if e.code == 429 and not ("PerDay" in msg or "limit: 0" in msg):
                        limited = True                      # sirf per-minute limit
                    else:
                        dead.add(model)                     # daily quota / 404 / 400
                        print(f"   {model} unusable ({e.code})")
                    break                                   # agla model try karo
                except Exception:                           # timeout / bad JSON
                    time.sleep(5)
        if all(m in dead for m in cands):
            return None, "ALL_DEAD"
        if limited and rnd == 0:
            print("   per-minute limit, 60s wait...")
            time.sleep(60)
        else:
            break
    return None, "FAILED"

# purane error entries hata do, sirf successful predictions rakho
old = json.load(open("predictions.json")) if os.path.exists("predictions.json") else {}
preds = {k: v for k, v in old.items() if "error" not in v}

for t in truth:
    sid = t["sample_id"]
    if sid in preds:
        print(sid, "cached"); continue
    path = f"samples/{sid}.pdf"
    if not os.path.exists(path):
        print(sid, "PDF missing, Cell 6 chalao"); continue
    result, info = extract_one(path)
    if result:
        preds[sid] = result
        json.dump(preds, open("predictions.json", "w"), indent=2)
        print(sid, "ok  ->", info)
    else:
        print(sid, "failed ->", info)
        if info == "ALL_DEAD":
            print("\nSab models ka quota khatam. Naya API key (naye project se) ya baad me retry karo.")
            break
    time.sleep(12)    # RPM limit se bachne ke liye

done = [t["sample_id"] for t in truth if t["sample_id"] in preds]
left = [t["sample_id"] for t in truth if t["sample_id"] not in preds]
print(f"\nOK: {len(done)} / {len(truth)}", "| baaki:", left if left else "none")
if not left:
    print("Sab ho gaya. Ab Cell 9 chalao.")

Cell 9: Compare + field_accuracy.csv


In [ ]:
import pandas as pd

def flat(d):
    out = {}
    for k, v in d.items():
        if k == "sample_id":
            continue
        if k == "vendor_address":
            for kk, vv in v.items():
                out[f"vendor_address.{kk}"] = vv
        elif k == "line_items":
            out["line_items_count"] = len(v)
            out["line_items"] = json.dumps([
                {kk: (round(float(vv), 2) if isinstance(vv, (int, float)) else str(vv).strip())
                 for kk, vv in sorted(li.items())} for li in v])
        else:
            out[k] = v
    return out

def norm(v):
    if v is None: return None
    if isinstance(v, (int, float)) and not isinstance(v, bool): return round(float(v), 2)
    return str(v).strip()

rows = []
for t in truth:
    sid = t["sample_id"]
    g = flat(t)
    p = flat(preds[sid]) if "error" not in preds[sid] else {}
    for field, gv in g.items():
        ev = p.get(field)
        rows.append({"sample_id": sid, "field_name": field,
                     "extracted_value": ev, "ground_truth_value": gv,
                     "exact_match": norm(ev) == norm(gv)})

df = pd.DataFrame(rows)
df.to_csv("field_accuracy.csv", index=False)

acc = (df.groupby("field_name")["exact_match"].mean() * 100).round(1).sort_values()
print(acc.to_string())
print("\nOverall:", round(df["exact_match"].mean() * 100, 1), "%")
df[~df["exact_match"]][["sample_id", "field_name", "extracted_value", "ground_truth_value"]]

Cell 10: accuracy_summary.md

In [ ]:
def err_type(e, g):
    if e is None and g is not None: return "missing (null instead of a value)"
    if g is None and e is not None: return "hallucinated (value for an absent field)"
    if isinstance(g, (int, float)) and isinstance(e, (int, float)): return "numeric mismatch"
    if isinstance(g, str) and len(g) == 10 and g[4:5] == "-": return "date mismatch"
    return "text mismatch (spelling/format)"

fails = df[~df["exact_match"]].copy()
fails["error_type"] = [err_type(norm(e), norm(g))
                       for e, g in zip(fails.extracted_value, fails.ground_truth_value)]
etypes = fails["error_type"].value_counts()

md = "# Accuracy Summary: Invoice Extractor\n\n"
md += f"Samples: {len(truth)} | Overall exact-match: {df['exact_match'].mean()*100:.1f}%\n\n"
md += "## Per-field accuracy\n\n| Field | Accuracy % |\n|---|---|\n"
for field, v in acc.items():
    md += f"| {field} | {v} |\n"

md += "\n## Which fields fail most\n\n"
worst = acc[acc < 100].head(3)
md += ("\n".join(f"- `{f}`: {v}%" for f, v in worst.items()) if len(worst) else "- All fields matched 100%") + "\n"

md += "\n## Common error types\n\n"
md += ("\n".join(f"- {k}: {v}" for k, v in etypes.items()) if len(etypes) else "- None") + "\n"

md += "\n## Failure details\n\n"
for _, r in fails.iterrows():
    md += f"- **{r.sample_id} / {r.field_name}** ({r.error_type}): got `{r.extracted_value}` vs expected `{r.ground_truth_value}`\n"

md += """
## Ideas to improve
- State date rules explicitly in the prompt (e.g. dd/mm/yyyy vs mm/dd/yyyy) and add few-shot examples.
- Add field descriptions in the schema for ambiguous fields (status, po_number).
- Validate in code: total_amount == subtotal + tax_amount; retry the call if it fails.
- Use fuzzy matching (or normalisation) for text fields during evaluation.
- Use a stronger model for failing fields only (cost vs accuracy trade-off).
"""
open("accuracy_summary.md", "w").write(md)
print(md)

Cell 11: README + zip download

In [ ]:
readme = """# Invoice Extractor (Gemini + Pydantic)

Extracts structured JSON from B2B invoice PDFs using Gemini multimodal + response_schema.

## Setup
pip install google-genai pydantic reportlab pandas
export GEMINI_API_KEY="your_key"

## Run
python main.py samples/s01.pdf

## Files
- schema.py: Pydantic model (nested Address, line_items list, status enum, optional po_number)
- llm.py: Gemini call wrapper (timeout, retry with backoff, model fallback)
- main.py: CLI, file_path -> JSON
- samples/: 10 synthetic invoice PDFs
- ground_truth.jsonl: expected output per sample
- field_accuracy.csv: per field per sample comparison
- accuracy_summary.md: per-field accuracy, error types, ideas to improve
"""
open("README.md", "w").write(readme)

import shutil
from google.colab import files
shutil.make_archive("/content/extractor_final", "zip", "/content/extractor")
files.download("/content/extractor_final.zip")